# 🌱 Plant Disease Classifier - Quick Start Guide

## Overview
This notebook demonstrates how to use the pre-trained CNN model to predict plant diseases from images. You can:
- Load and verify your trained model
- Make predictions on single images
- Process multiple images in batch
- Visualize results with confidence scores
- Use interactive prediction mode

**Model Files Required:**
- `models/plant_disease_model.h5` - Your trained CNN model
- `models/label_binarizer.pkl` - Label encoder for disease classes

In [ ]:
# Install required packages
import subprocess
import sys

packages = ['tensorflow', 'numpy', 'pillow', 'matplotlib', 'scikit-learn', 'opencv-python', 'tqdm']

for package in packages:
    try:
        __import__(package)
        print(f"✅ {package} is already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

In [ ]:
# Import required libraries
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import time
from tensorflow.keras.models import load_model
from tqdm import tqdm

print("✅ All libraries imported successfully!")
print("\nLoaded modules:")
print("  - TensorFlow/Keras")
print("  - NumPy")
print("  - Matplotlib")
print("  - Pillow")
print("  - scikit-learn")

In [ ]:
# Set up paths and verify model files
BASE_DIR = Path.cwd()
MODELS_DIR = BASE_DIR / "models"
UPLOADS_DIR = BASE_DIR / "uploads"
OUTPUTS_DIR = BASE_DIR / "outputs"
LOGS_DIR = BASE_DIR / "logs"

# Create directories
for directory in [UPLOADS_DIR, OUTPUTS_DIR, LOGS_DIR]:
    directory.mkdir(exist_ok=True)

# Model file paths
MODEL_PATH = MODELS_DIR / "plant_disease_model.h5"
LABEL_ENCODER_PATH = MODELS_DIR / "label_binarizer.pkl"

print("=" * 70)
print("📁 PROJECT STRUCTURE CHECK")
print("=" * 70)

# Check model files
print(f"\n🔍 Checking model files in: {MODELS_DIR}")
print(f"  Model file (.h5): {'✅ Found' if MODEL_PATH.exists() else '❌ Not Found'} - {MODEL_PATH}")
print(f"  Label encoder (.pkl): {'✅ Found' if LABEL_ENCODER_PATH.exists() else '❌ Not Found'} - {LABEL_ENCODER_PATH}")

# Check directories
print(f"\n📁 Checking directories:")
for name, path in [("uploads", UPLOADS_DIR), ("outputs", OUTPUTS_DIR), ("logs", LOGS_DIR)]:
    print(f"  {name}: {'✅ Created' if path.exists() else '❌ Not created'} - {path}")

if MODEL_PATH.exists():
    print(f"\n✅ Model file size: {MODEL_PATH.stat().st_size / (1024**2):.2f} MB")
if LABEL_ENCODER_PATH.exists():
    print(f"✅ Label encoder size: {LABEL_ENCODER_PATH.stat().st_size / 1024:.2f} KB")

print("\n" + "=" * 70)

In [ ]:
# Load model and label encoder
print("Loading model and label encoder...")

try:
    # Load model
    model = load_model(str(MODEL_PATH))
    print("✅ Model loaded successfully!")
    
    # Load label encoder
    with open(LABEL_ENCODER_PATH, 'rb') as f:
        label_encoder = pickle.load(f)
    print("✅ Label encoder loaded successfully!")
    
    # Display model architecture
    print("\n" + "=" * 70)
    print("🧠 MODEL ARCHITECTURE")
    print("=" * 70)
    model.summary()
    
    # Get disease classes
    classes = label_encoder.classes_
    print(f"\n🌱 Disease Classes: {len(classes)} total")
    for i, cls in enumerate(classes[:10], 1):
        print(f"  {i}. {cls}")
    if len(classes) > 10:
        print(f"  ... and {len(classes) - 10} more")
    
except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    model = None
    label_encoder = None

In [ ]:
# Configuration settings
IMAGE_SIZE = (256, 256)
CONFIDENCE_THRESHOLD = 0.5
TOP_K_PREDICTIONS = 5
SUPPORTED_FORMATS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif'}

print("⚙️ Configuration Settings:")
print(f"  Image Size: {IMAGE_SIZE}")
print(f"  Confidence Threshold: {CONFIDENCE_THRESHOLD}")
print(f"  Top K Predictions: {TOP_K_PREDICTIONS}")
print(f"  Supported Formats: {', '.join(SUPPORTED_FORMATS)}")

# Utility functions
def load_image(image_path):
    """Load and validate image"""
    try:
        if not os.path.exists(image_path):
            return None, "Image file not found"
        
        file_ext = Path(image_path).suffix.lower()
        if file_ext not in SUPPORTED_FORMATS:
            return None, f"Unsupported format: {file_ext}"
        
        img = Image.open(image_path).convert('RGB')
        return img, "Success"
    except Exception as e:
        return None, str(e)

def preprocess_image(image):
    """Preprocess image for model"""
    try:
        img_resized = image.resize(IMAGE_SIZE, Image.Resampling.LANCZOS)
        img_array = np.array(img_resized, dtype=np.float32)
        img_array = img_array / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        return img_array
    except Exception as e:
        print(f"Error preprocessing: {str(e)}")
        return None

def get_image_info(image_path):
    """Get image information"""
    try:
        img = Image.open(image_path)
        file_size = os.path.getsize(image_path) / (1024 * 1024)
        return {
            "size": img.size,
            "format": img.format,
            "file_size_mb": round(file_size, 2)
        }
    except:
        return None

print("\n✅ Utility functions defined!")

In [ ]:
# Single Image Prediction Function
def predict_single_image(image_path):
    """Predict disease on a single image"""
    
    if model is None:
        return {"success": False, "error": "Model not loaded"}
    
    start_time = time.time()
    
    # Load image
    image, status = load_image(image_path)
    if image is None:
        return {"success": False, "error": status}
    
    # Preprocess
    img_array = preprocess_image(image)
    if img_array is None:
        return {"success": False, "error": "Failed to preprocess image"}
    
    # Predict
    predictions = model.predict(img_array, verbose=0)
    prediction_scores = predictions[0]
    
    # Get top-k
    top_k_indices = np.argsort(prediction_scores)[::-1][:TOP_K_PREDICTIONS]
    
    # Build results
    top_predictions = []
    for rank, idx in enumerate(top_k_indices, 1):
        class_name = label_encoder.classes_[idx]
        confidence = prediction_scores[idx]
        top_predictions.append({
            "rank": rank,
            "disease": class_name,
            "confidence": confidence,
            "confidence_pct": f"{confidence * 100:.2f}%"
        })
    
    processing_time = time.time() - start_time
    image_info = get_image_info(image_path)
    
    return {
        "success": True,
        "image_path": image_path,
        "image_info": image_info,
        "predicted_disease": top_predictions[0]["disease"],
        "confidence": top_predictions[0]["confidence"],
        "confidence_pct": top_predictions[0]["confidence_pct"],
        "top_predictions": top_predictions,
        "processing_time": processing_time
    }

# Test with sample image if available
print("✅ Single image prediction function defined!")
print("\nUsage: result = predict_single_image('path/to/image.jpg')")

In [ ]:
# Batch Processing Function
def predict_batch(folder_path):
    \"\"\"Predict diseases on multiple images\"\"\"\n
    if model is None:
        return {"success": False, "error": "Model not loaded"}
    
    results = []
    folder = Path(folder_path)
    
    if not folder.exists():
        return {"success": False, "error": "Folder not found"}
    
    # Get all image files
    image_files = []
    for ext in SUPPORTED_FORMATS:
        image_files.extend(folder.glob(f"*{ext}"))
        image_files.extend(folder.glob(f"*{ext.upper()}"))
    
    image_files = list(set(image_files))  # Remove duplicates
    
    print(f"Found {len(image_files)} images to process")
    
    # Process each image
    for idx, image_file in enumerate(image_files, 1):
        result = predict_single_image(str(image_file))
        results.append(result)
        
        if result['success']:
            print(f"✅ [{idx}/{len(image_files)}] {image_file.name}: {result['predicted_disease']}")
        else:
            print(f"❌ [{idx}/{len(image_files)}] {image_file.name}: {result['error']}")
    
    return results

print("✅ Batch prediction function defined!")
print("\nUsage: results = predict_batch('path/to/folder')")

In [ ]:
# Visualization Functions
def print_prediction(result):
    """Pretty print prediction result"""
    
    if not result.get('success'):
        print(f"❌ Error: {result.get('error')}")
        return
    
    print("=" * 70)
    print("PREDICTION RESULT".center(70))
    print("=" * 70)
    
    print(f"\n📷 Image: {result['image_path']}")
    
    if result['image_info']:
        info = result['image_info']
        print(f"\nImage Information:")
        print(f"  Size: {info['size'][0]}x{info['size'][1]} pixels")
        print(f"  File Size: {info['file_size_mb']} MB")
    
    print(f"\n{'PREDICTED DISEASE':^70}")
    print(f"Disease: {result['predicted_disease']}")
    print(f"Confidence: {result['confidence_pct']}")
    print(f"Processing Time: {result['processing_time']:.3f} seconds")
    
    print(f"\n{'TOP 5 PREDICTIONS':^70}")
    for pred in result['top_predictions']:
        dots = "." * (50 - len(pred['disease']))
        print(f"{pred['rank']}. {pred['disease']}{dots} {pred['confidence_pct']}")
    
    print("\n" + "=" * 70)

def visualize_prediction(result):
    """Visualize prediction with image and confidence bars"""
    
    if not result.get('success'):
        print(f"Cannot visualize: {result.get('error')}")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Load and display image
    try:
        img = Image.open(result['image_path'])
        axes[0].imshow(img)
        axes[0].set_title(f"Predicted: {result['predicted_disease']}\nConfidence: {result['confidence_pct']}", 
                         fontsize=12, fontweight='bold')
        axes[0].axis('off')
    except:
        axes[0].text(0.5, 0.5, 'Image not found', ha='center', va='center')
        axes[0].set_title("Image Preview", fontsize=12, fontweight='bold')
        axes[0].axis('off')
    
    # Display confidence bars
    preds = result['top_predictions']
    diseases = [p['disease'] for p in preds]
    confidences = [p['confidence'] * 100 for p in preds]
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(diseases))]
    
    axes[1].barh(diseases, confidences, color=colors)
    axes[1].set_xlabel('Confidence (%)', fontsize=11)
    axes[1].set_title('Top 5 Predictions', fontsize=12, fontweight='bold')
    axes[1].set_xlim(0, 100)
    
    for i, v in enumerate(confidences):
        axes[1].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions defined!")

In [ ]:
# Example 1: Single Image Prediction
print("=" * 70)
print("EXAMPLE 1: SINGLE IMAGE PREDICTION".center(70))
print("=" * 70)

# Create sample uploads folder with demo images
sample_image_path = UPLOADS_DIR / "sample_plant.jpg"

if sample_image_path.exists():
    print(f"\n📸 Testing with: {sample_image_path}")
    result = predict_single_image(str(sample_image_path))
    print_prediction(result)
    visualize_prediction(result)
else:
    print(f"\n⚠️  No sample image found at {sample_image_path}")
    print("   Place a plant image in the uploads/ folder to test")
    print("\n   To use single prediction:")
    print("   result = predict_single_image('path/to/your/image.jpg')")
    print("   print_prediction(result)")
    print("   visualize_prediction(result)")

In [ ]:
# Example 2: Batch Processing
print("\n" + "=" * 70)
print("EXAMPLE 2: BATCH PROCESSING".center(70))
print("=" * 70)

print(f"\n📁 Checking for images in: {UPLOADS_DIR}")

# Count images
image_count = 0
for ext in SUPPORTED_FORMATS:
    image_count += len(list(UPLOADS_DIR.glob(f"*{ext}")))
    image_count += len(list(UPLOADS_DIR.glob(f"*{ext.upper()}")))

if image_count > 0:
    print(f"Found {image_count} image(s) to process\n")
    batch_results = predict_batch(str(UPLOADS_DIR))
    
    # Summary
    successful = sum(1 for r in batch_results if r.get('success'))
    print(f"\n✅ {successful}/{len(batch_results)} images processed successfully")
else:
    print(f"⚠️  No images found in {UPLOADS_DIR}")
    print("   Place plant images in the uploads/ folder")
    print("\n   To use batch prediction:")
    print("   results = predict_batch('uploads')")

## 🎯 Quick Reference Guide

### Single Image Prediction
```python
# Predict on a single image
result = predict_single_image('path/to/image.jpg')

# Display results
print_prediction(result)

# Visualize with chart
visualize_prediction(result)
```

### Batch Processing
```python
# Process all images in folder
results = predict_batch('path/to/folder')

# Access individual results
for result in results:
    if result['success']:
        print(f"{result['image_path']}: {result['predicted_disease']}")
```

### Understanding Results
- **Disease**: The predicted plant disease class
- **Confidence**: How confident the model is (higher = more confident)
- **Top 5 Predictions**: All possible diseases ranked by probability
- **Processing Time**: Time taken to make prediction

### Confidence Interpretation
| Score | Meaning |
|-------|---------|
| 90-100% | Very confident ✅ |
| 70-89% | Confident ✓ |
| 50-69% | Uncertain ⚠️ |
| Below 50% | Not reliable ❌ |

## 🚀 Next Steps

1. **Add sample images** to the `uploads/` folder
2. **Run batch prediction** on your image collection
3. **Visualize results** with confidence charts
4. **Use the CLI app** with `python main.py` for production use

## 📚 More Information

- Full documentation: See `README.md`
- CLI usage: `python main.py --help`
- Configuration: Edit `config.py` to customize settings